# ERK-KTR Full FOV Stimulation Pipeline (RTMSequence)
This notebook uses RTMSequence to define acquisition events instead of the legacy generate_df_acquire() workflow.

In [ ]:
import os
import time
from rtm_pymmcore.core.data_structures import Channel, PowerChannel, RTMSequence, SegmentationMethod
import rtm_pymmcore.core.utils as utils

### Experimental Settings

In [ ]:
from rtm_pymmcore.microscope.pertzlab.jungfrau import Jungfrau

mic = Jungfrau()
mic.mmc.setChannelGroup("TTL_ERK")

In [ ]:
## Configuration
SLEEP_BEFORE_EXPERIMENT_START_in_H = 0

## Storage
base_path = "E:\\Alex"
experiment_name = "2025-11-13_Priming_FGFR_U0126"
path = os.path.join(base_path, experiment_name)

## Stimulation channel
stim_channel = PowerChannel(
    config="CyanStim", exposure=100, group="TTL_ERK",
    power=10,
)

## Optocheck channel
optocheck_channel = Channel(config="mCitrine", exposure=600)

### Pipeline Setup

In [ ]:
from rtm_pymmcore.stimulation.base import StimWholeFOV
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.erk_ktr import FE_ErkKtr
from rtm_pymmcore.feature_extraction.optocheck import OptoCheckFE
from rtm_pymmcore.segmentation.remote import SegmentatorImagingServerKit

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=SegmentatorImagingServerKit(
            server="http://izbniesen.izb.unibe.ch:8001",
            algorithm="StarDist (2D)",
            min_size=50,
            model_param={"model_name": "MCF10A_v01"},
        ),
        use_channel=0,
        save_tracked=True,
    )
]

stimulator = StimWholeFOV()
feature_extractor = FE_ErkKtr("labels")
tracker = TrackerTrackpy(search_range=50)
optocheck = OptoCheckFE(used_mask="labels")

from rtm_pymmcore.core.pipeline import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_optocheck=optocheck,
)
mic.set_pipeline(pipeline=pipeline)

### GUI

In [ ]:
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)
data_mda_fovs = None
load_from_file = False

In [ ]:
if mic.SET_ROI_REQUIRED:
    mic.mmc.clearROI()
    mic.mmc.setROI(mic.ROI_X, mic.ROI_Y, mic.ROI_WIDTH, mic.ROI_HEIGHT)

### Build acquisition events

In [ ]:
fov_positions = utils.generate_fov_positions(mic, viewer=viewer)

phase_1 = RTMSequence(
    time_plan={"interval": 60.0, "loops": 100},
    stage_positions=fov_positions,
    channels=[{"config": "miRFP", "exposure": 300}, {"config": "mScarlet3", "exposure": 200}],
    stim_channels=(stim_channel,),
    stim_frames=range(10, 100),
    rtm_metadata={"phase_name": "PreDrug", "phase_id": 0, "treatment_name": "Priming Phase 1 pre Drug"},
)

phase_2 = RTMSequence(
    time_plan={"interval": 60.0, "loops": 150},
    stage_positions=fov_positions,
    channels=[{"config": "miRFP", "exposure": 300}, {"config": "mScarlet3", "exposure": 200}],
    stim_channels=(stim_channel,),
    stim_frames=range(30, 120),
    optocheck_channels=(optocheck_channel,),
    optocheck_frames={149},
    rtm_metadata={"phase_name": "PostDrug", "phase_id": 1, "treatment_name": "Sustained Phase 2 post Drug"},
)

events = phase_1 + phase_2

print(f"Total events: {len(events)}")
utils.events_to_dataframe(events)

In [ ]:
mic.validate_events(events)

### Run experiment

In [ ]:
for _ in range(0, int(SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600)):
    time.sleep(1)

try:
    mm_wdg._core_link.cleanup()
except:
    pass

mic.run_experiment(events)

In [ ]:
mic.post_experiment()
time.sleep(10)

utils.generate_exp_data_from_tracks(path)

from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)